### Mugrade boilerplate

In [ ]:
### Run this cell to download and installs the necessary modules for the homework
%pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part1_tokenizer_tests.py

import mugrade
import os
from part1_tokenizer_tests import *
os.environ["MUGRADE_HW"] = "Part 1 - Tokenizer"
# Read the Mugrade API key from the environment or from a local .env file.
# Never hard-code the key. See .env.example in the repository root.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if not os.environ.get("MUGRADE_KEY"):
    print("MUGRADE_KEY is not set. Cells that use @mugrade.submit_tests need it.")

### BPE Training, Encoding, Decoding

For this portion of the homework, implement the BPE class we covered below.  You don't need to run the later pieces of code (they actually train the tokenize on the FineWeb-EDU dataset), but you are free to do so if you want to train your own tokenizer.  For your submission, you should filter word frequencies in training to ignore words that only occur once (we did both versions in class, but the test cases require the `train` function to filter words in that manner.)

Note that the mugrade local tests are commented by default, but you should uncomment them as you implement each function.  Unlike the previous assignment, we know that the details of each function likely won't make much sense without watching the lecture, but after watching the lecture the form of the function should be clear.

In [4]:
import json
from collections import Counter
import re
from tqdm.auto import tqdm

class BPE:
    @mugrade.submit_tests
    def __init__(self, filename=None):
        ### BEGIN YOUR CODE
        if filename is not None:
            with open(filename, "rt") as fobj:
                self.vocab, merge_keys, merge_values, self.special_tokens = json.load(fobj)
                self.merges = {
                    tuple(k): v for k, v in zip(merge_keys, merge_values)
                }
                self.special_tokens = {
                    int(k): v for k, v in self.special_tokens.items()
                }
        else:
            self.vocab = [chr(i) for i in range(256)]
            self.merges = {}
            self.special_tokens = {}
        ### END YOUR CODE

    @staticmethod
    @mugrade.submit_tests
    def merge_pair(word, a, b, merged):
        ### BEGIN YOUR CODE
        i = 0
        while i < len(word) - 1:
            if word[i] == a and word[i+1] == b:
                word[i: i+2] = [merged]
            i+=1
        ### END YOUR CODE

    @mugrade.submit_tests
    def replace_special_tokens(self, word):
        ### BEGIN YOUR CODE
        for k, v in self.special_tokens.items():
            word = word.replace(v, chr(k))
        return word
        ### END YOUR CODE


    @mugrade.submit_tests
    def train(self, text, target_vocab_size):
        ### BEGIN YOUR CODE
        regex_matching = re.findall(r"\s\S+|\s+|\S+", text)
        frequency_counter = Counter(regex_matching)
        frequency_map = {
            self.replace_special_tokens(k): v for k, v in frequency_counter.items() if v > 1
        }

        # Build word_list_frequency_map without defaultdict
        word_list_frequency_map = {}
        
        for word, freq in frequency_map.items():
            filtered_word = tuple(
                ord(char) for char in word 
                if ord(char) not in self.special_tokens
            )
            
            if filtered_word:
                word_list_frequency_map[filtered_word] = \
                    word_list_frequency_map.get(filtered_word, 0) + freq

        words_as_lists, words_as_lists_frequencies = \
            [list(i) for i in word_list_frequency_map.keys()], list(word_list_frequency_map.values())


        for _ in range(len(self.vocab), target_vocab_size):
            existing_vocab_size = len(self.vocab)
            suitable_merge_pairs = Counter()

            for current_word_as_list, current_word_frequency in \
                zip(words_as_lists, words_as_lists_frequencies):

                for i in range(len(current_word_as_list) - 1):
                    suitable_merge_pairs[(current_word_as_list[i], current_word_as_list[i+1])] +=\
                        current_word_frequency

            # Break condition for testing
            # if len(suitable_merge_pairs) == 0:
            #     break

            symbol_a, symbol_b = suitable_merge_pairs.most_common()[0][0]
            self.vocab.append(self.vocab[symbol_a] + self.vocab[symbol_b])
            self.merges[(symbol_a, symbol_b)] = existing_vocab_size

            for word_as_list in words_as_lists:
                self.merge_pair(word_as_list, symbol_a, symbol_b, existing_vocab_size)

        if len(self.special_tokens) != 0:
            max_special_token = max(self.special_tokens.keys())
            self.vocab += [""] * (max_special_token - len(self.vocab) + 1)
            for k, v in self.special_tokens.items():
                self.vocab[k] = v
        ### END YOUR CODE

    @mugrade.submit_tests
    def encode(self, text):
        ### BEGIN YOUR CODE
        tokens = []

        for word in re.findall(r"\s\S+|\s+|\S+", text):
            word_as_list = [ord(char) for char in self.replace_special_tokens(word)]
            # Special tokens are now single code points (e.g. chr(32760)). No merge
            # contains them, so they pass through as their own token IDs.
            
            if  len(word_as_list) == 0:
                continue

            while True:
                word_symbol_pairs_to_vocab_index_mapping = {}
                for i in range(len(word_as_list) - 1):
                    current_symbol_pair = \
                        (word_as_list[i], word_as_list[i+1])
                    if current_symbol_pair in self.merges:
                        word_symbol_pairs_to_vocab_index_mapping[current_symbol_pair] = \
                            self.merges[current_symbol_pair]

                if len(word_symbol_pairs_to_vocab_index_mapping) == 0:
                    break

                symbol_a, symbol_b = min_symbol_pair = min(
                    word_symbol_pairs_to_vocab_index_mapping,
                    key = word_symbol_pairs_to_vocab_index_mapping.get
                )
                self.merge_pair(word_as_list, symbol_a, symbol_b, 
                                word_symbol_pairs_to_vocab_index_mapping[min_symbol_pair])

            tokens += word_as_list

        return tokens
        ### END YOUR CODE

    @mugrade.submit_tests
    def decode(self, tokens):
        ### BEGIN YOUR CODE
        return "".join([self.vocab[token] for token in tokens])
        ### END YOUR CODE

    @mugrade.submit_tests
    def save(self, fname):
        ### BEGIN YOUR CODE
        with open(fname, "wt") as fobj:
            json.dump([
                self.vocab,
                list(self.merges.keys()),
                list(self.merges.values()),
                self.special_tokens
            ], fobj)
        ### END YOUR CODE

Mugrade: Submitting tests for function __init__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Grader test 6 passed
Mugrade: Submitting tests for function merge_pair():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function replace_special_tokens():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Mugrade: Submitting tests for function train():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Mugrade: Submitting tests for function encode():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Mugrade: Submitting tests for function decode():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Mugrade: Submitting tests for function save():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed


In [ ]:
# Sanity check: train a tiny tokenizer with one special token, then confirm that
# decode(encode(text)) returns each input text unchanged, special token included.
samples = [
    "<doc>Hey hey hey hey there lads lads and ladies",
    "<doc> Hey hey hey hey there lads lads and ladies",
    "Hey hey<doc> hey hey there lads lads and ladies",
    "Hey hey hey hey there lads lads and ladies <doc>",
]

my_bpe = BPE()
my_bpe.special_tokens[260] = "<doc>"
my_bpe.train(samples[0], 260)

for sample in samples:
    print(my_bpe.decode(my_bpe.encode(sample)))

### Download datasets

The following code with download the (10B token subset of the) FineWeb-EDU dataset, plus the SmolTalk dataset.  To make this suitably fast, you shuld really get put a huggingface token in your .env file or equivalent for the notebook.

In [ ]:
from huggingface_hub import snapshot_download
import pyarrow.dataset as ds
import random
from dotenv import load_dotenv; load_dotenv()
import os
os.environ["HF_XET_HIGH_PERFORMANCE"]="1"

def download_dataset(name, subdir, field, output_file, chat=False):
    root = snapshot_download(name, repo_type="dataset", allow_patterns=subdir+"/*.parquet", max_workers=100)
    docs = ds.dataset(root + "/" + subdir, format="parquet").to_table()[field].to_pylist()
    random.seed(42)
    random.shuffle(docs)

    with open(output_file, "wt") as f:
        for doc in tqdm(docs):
            if chat:
                msgs = "".join([f"<{m["role"].upper()}>{m["content"]}</{m["role"].upper()}>" for m in doc])
                f.write(f"<DOCUMENT>{msgs}</DOCUMENT>")
            else:
                f.write(f"<DOCUMENT>{doc}</DOCUMENT>")

In [ ]:
download_dataset("HuggingFaceFW/fineweb-edu", "sample/10BT", "text", "fineweb-edu-10BT.shuffle2.txt")
download_dataset("HuggingFaceTB/smoltalk", "data/all", "messages", "smoltalk.shuffle.txt", chat=True)

### Train the tokenizer

Once you've downloaded the datasets, these next cells with train the tokenizer.  Note that this takes a long time, e.g. on my machine it takes about an hour.

In [ ]:
with open("fineweb-edu-10BT.shuffle2.txt", "rb") as f:
    text_train = f.read(40_000_000).decode("latin-1")
with open("smoltalk.shuffle.txt", "rb") as f:
    text_train += f.read(10_000_000).decode("latin-1")

bpe = BPE()
vocab_size = 2**15-256
bpe.special_tokens[vocab_size] = "<DOCUMENT>"
bpe.special_tokens[vocab_size+1] = "</DOCUMENT>"
bpe.special_tokens[vocab_size+2] = "<SYSTEM>"
bpe.special_tokens[vocab_size+3] = "</SYSTEM>"
bpe.special_tokens[vocab_size+4] = "<USER>"
bpe.special_tokens[vocab_size+5] = "</USER>"
bpe.special_tokens[vocab_size+6] = "<ASSISTANT>"
bpe.special_tokens[vocab_size+7] = "</ASSISTANT>"

bpe.train(text_train, vocab_size)
bpe.save("tokenizer_50M.bpe")

### Pretokenize the datasets

Finally, you can use the tokenizer to tokenize the entirety of our text datasets.  Since we'll be using them a lot during training, it's a good idea to take this approach and pre-convert the datasets to tokens that we can use directly for training, rather than doing the tokenization in real time.  This also takes a relatively long time, but since it is trivially parallelizable, if you have a machine with a lot of CPUs it can be done quickly for the entire datasets.

In [ ]:
from joblib import Parallel, delayed
from array import array
import os

def pretokenize(in_fname, out_fname, bpe, chunk_size = 20_000_000, tmpdir=".tmp"):
    os.makedirs(tmpdir, exist_ok=True)

    def tokenize_chunk(i):
        with open(in_fname, "rb") as f:
            f.seek(i*chunk_size)
            tokens = bpe.encode(f.read(chunk_size).decode("latin-1"))
        with open(f"{tmpdir}/tokens.{i:05d}.bin", "wb") as f:
            array("H", tokens).tofile(f)

    n_chunks = (os.path.getsize(in_fname) + chunk_size - 1) // chunk_size
    jobs = Parallel(n_jobs=128, return_as="generator_unordered")(
        delayed(tokenize_chunk)(i) for i in range(n_chunks)
    )
    for _ in tqdm(jobs, total=n_chunks, smoothing=0):
        pass

    with open(out_fname, "wb") as fout:
        for i in tqdm(range(n_chunks)):
            with open(f"{tmpdir}/tokens.{i:05d}.bin", "rb") as fin:
                fout.write(fin.read())

In [ ]:
bpe = BPE("tokenizer_50M.bpe")
pretokenize("smoltalk.shuffle.txt", "smoltalk.shuffle.bin", bpe)
pretokenize("fineweb-edu-10BT.shuffle2.txt", "fineweb-edu-10BT.shuffle.bin", bpe)